In [1]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ["notebooks", "chapter09", "chapter10"] else CURRENT_DIR

print("현재 폴더 이름:", CURRENT_DIR.name)
print("파이썬 버전:", sys.version.split()[0])

for name in ["orders.csv", "order_items.csv", "customers.csv"]:
    exists = (PROJECT_ROOT / "data" / "raw" / name).exists()
    print(name, "->", "있음" if exists else "없음")

현재 폴더 이름: chapter10
파이썬 버전: 3.14.6
orders.csv -> 있음
order_items.csv -> 있음
customers.csv -> 있음


### STEP 1. Target Contract와 예측 시점 정의

- **무엇을 예측하는가?** 주문이 취소될지 여부
- **예측 단위**: 주문 1건
- **언제 예측하는가?** 주문이 생성된 직후
- **Target**: `is_cancelled` (completed = 0, cancelled = 1)
- **제외**: refunded와 기타 상태는 모델링 범위에서 뺀다.
- **알 수 있는 정보**: 주문 월·요일, 결제수단, 상품 수량·금액, 고객 나이·성별·도시
- **쓰면 안 되는 정보**: `order_status`, `is_cancelled`, ID(`order_id`, `customer_id`, `product_id`), 취소 사유와 취소 시각 같은 사후 정보
- 파일에 있는 컬럼이라도 예측 시점에 알 수 없으면 feature로 쓰지 않는다.

In [2]:
import pandas as pd
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ["notebooks", "chapter09", "chapter10"] else CURRENT_DIR
DATA = PROJECT_ROOT / "data"

def load(path):
    return pd.read_csv(path, encoding="utf-8-sig") if path.exists() else None

files = {
    "raw/orders": DATA / "raw" / "orders.csv",
    "raw/order_items": DATA / "raw" / "order_items.csv",
    "raw/customers": DATA / "raw" / "customers.csv",
    "processed/order_items_clean": DATA / "processed" / "order_items_clean.csv",
    "processed/customers_clean": DATA / "processed" / "customers_clean.csv",
    "data/orders_clean": DATA / "orders_clean.csv",
}
dfs = {k: load(p) for k, p in files.items()}

# 1) 파일별 구조 (행/열, 컬럼, 결측, 중복)
for name, df in dfs.items():
    print("=" * 50)
    if df is None:
        print(name, "-> 파일 없음")
        continue
    print(name, "| 행:", df.shape[0], "| 열:", df.shape[1])
    print("컬럼:", list(df.columns))
    print("결측치 있는 컬럼:", df.isna().sum()[df.isna().sum() > 0].to_dict())
    print("완전 중복 행:", int(df.duplicated().sum()))

# 2) 관계 확인 (연결 안 되는 행이 있는지)
def orphan(child, key, parent):
    if child is None or parent is None or key not in child or key not in parent:
        return "확인 불가"
    return int((~child[key].isin(parent[key])).sum())

print("=" * 50)
print("[raw] order_items에 있는데 orders에 없는 주문:",
      orphan(dfs["raw/order_items"], "order_id", dfs["raw/orders"]))
print("[raw] orders에 있는데 customers에 없는 고객:",
      orphan(dfs["raw/orders"], "customer_id", dfs["raw/customers"]))
print("[processed] order_items에 있는데 orders_clean에 없는 주문:",
      orphan(dfs["processed/order_items_clean"], "order_id", dfs["data/orders_clean"]))
print("[processed] orders_clean에 있는데 customers_clean에 없는 고객:",
      orphan(dfs["data/orders_clean"], "customer_id", dfs["processed/customers_clean"]))

# 3) 타깃 후보 (주문 상태)
for key in ["raw/orders", "data/orders_clean"]:
    df = dfs[key]
    if df is not None and "order_status" in df:
        print("=" * 50)
        print(key, "order_status 건수")
        print(df["order_status"].value_counts(dropna=False))

raw/orders | 행: 300 | 열: 5
컬럼: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
결측치 있는 컬럼: {}
완전 중복 행: 0
raw/order_items | 행: 764 | 열: 5
컬럼: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']
결측치 있는 컬럼: {}
완전 중복 행: 0
raw/customers | 행: 150 | 열: 6
컬럼: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
결측치 있는 컬럼: {}
완전 중복 행: 0
processed/order_items_clean | 행: 752 | 열: 6
컬럼: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']
결측치 있는 컬럼: {}
완전 중복 행: 0
processed/customers_clean | 행: 150 | 열: 6
컬럼: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
결측치 있는 컬럼: {'signup_date': 2}
완전 중복 행: 0
data/orders_clean | 행: 300 | 열: 7
컬럼: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
결측치 있는 컬럼: {'order_date': 3, 'order_month': 3, 'order_dayofweek': 3}
완전 중복 행: 0
[raw] order_items에 있는데 orders에 없는 주문: 0
[raw] orders에 있는데 customers에 없는 고객: 0


In [3]:
import pandas as pd
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ["notebooks", "chapter09", "chapter10"] else CURRENT_DIR
DATA = PROJECT_ROOT / "data"

paths = {
    "orders": DATA / "orders_clean.csv",
    "order_items": DATA / "processed" / "order_items_clean.csv",
    "customers": DATA / "processed" / "customers_clean.csv",
}

for name, path in paths.items():
    print("=" * 50)
    if not path.exists():
        print(name, "-> 파일 없음")
        continue
    df = pd.read_csv(path, encoding="utf-8-sig")
    print(name, "| 행:", df.shape[0], "| 열:", df.shape[1])
    print()
    print("[컬럼과 자료형]")
    print(df.dtypes.to_string())
    print()
    missing = df.isna().sum()
    print("[결측치 있는 컬럼]", missing[missing > 0].to_dict() or "없음")
    print("[완전 중복 행]", int(df.duplicated().sum()))
    obj_cols = df.select_dtypes(include="object").columns
    for c in obj_cols:
        if df[c].nunique() <= 10:
            print(f"[{c} 값 종류]", df[c].value_counts(dropna=False).to_dict())

orders | 행: 300 | 열: 7

[컬럼과 자료형]
order_id           int64
customer_id        int64
order_date           str
payment_method       str
order_status         str
order_month          str
order_dayofweek      str

[결측치 있는 컬럼] {'order_date': 3, 'order_month': 3, 'order_dayofweek': 3}
[완전 중복 행] 0
[payment_method 값 종류] {'kakao_pay': 90, 'card': 72, 'bank_transfer': 70, 'naver_pay': 68}
[order_status 값 종류] {'completed': 180, 'cancelled': 63, 'refunded': 57}
[order_dayofweek 값 종류] {'Wednesday': 58, 'Thursday': 49, 'Tuesday': 44, 'Friday': 43, 'Monday': 40, 'Sunday': 35, 'Saturday': 28, nan: 3}
order_items | 행: 752 | 열: 6

[컬럼과 자료형]
order_item_id      int64
order_id           int64
product_id         int64
quantity         float64
unit_price       float64
line_total       float64

[결측치 있는 컬럼] 없음
[완전 중복 행] 0
customers | 행: 150 | 열: 6

[컬럼과 자료형]
customer_id      int64
name               str
gender             str
age            float64
city               str
signup_date        str

[결측치 있는 컬럼] {'s

C:\Users\apll1\AppData\Local\Temp\ipykernel_32024\290482630.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include="object").columns
C:\Users\apll1\AppData\Local\Temp\ipykernel_32024\290482630.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

In [4]:
import pandas as pd
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ["notebooks", "chapter09", "chapter10"] else CURRENT_DIR
DATA = PROJECT_ROOT / "data"

paths = {
    "orders": DATA / "orders_clean.csv",
    "order_items": DATA / "processed" / "order_items_clean.csv",
    "customers": DATA / "processed" / "customers_clean.csv",
}

for name, path in paths.items():
    print("=" * 50)
    if not path.exists():
        print(name, "-> 파일 없음")
        continue
    df = pd.read_csv(path, encoding="utf-8-sig")
    print(name, "| 행:", df.shape[0], "| 열:", df.shape[1])
    print()
    print("[컬럼과 자료형]")
    print(df.dtypes.to_string())
    print()
    missing = df.isna().sum()
    print("[결측치 있는 컬럼]", missing[missing > 0].to_dict() or "없음")
    print("[완전 중복 행]", int(df.duplicated().sum()))
    obj_cols = df.select_dtypes(include="object").columns
    for c in obj_cols:
        if df[c].nunique() <= 10:
            print(f"[{c} 값 종류]", df[c].value_counts(dropna=False).to_dict())

orders | 행: 300 | 열: 7

[컬럼과 자료형]
order_id           int64
customer_id        int64
order_date           str
payment_method       str
order_status         str
order_month          str
order_dayofweek      str

[결측치 있는 컬럼] {'order_date': 3, 'order_month': 3, 'order_dayofweek': 3}
[완전 중복 행] 0
[payment_method 값 종류] {'kakao_pay': 90, 'card': 72, 'bank_transfer': 70, 'naver_pay': 68}
[order_status 값 종류] {'completed': 180, 'cancelled': 63, 'refunded': 57}
[order_dayofweek 값 종류] {'Wednesday': 58, 'Thursday': 49, 'Tuesday': 44, 'Friday': 43, 'Monday': 40, 'Sunday': 35, 'Saturday': 28, nan: 3}
order_items | 행: 752 | 열: 6

[컬럼과 자료형]
order_item_id      int64
order_id           int64
product_id         int64
quantity         float64
unit_price       float64
line_total       float64

[결측치 있는 컬럼] 없음
[완전 중복 행] 0
customers | 행: 150 | 열: 6

[컬럼과 자료형]
customer_id      int64
name               str
gender             str
age            float64
city               str
signup_date        str

[결측치 있는 컬럼] {'s

C:\Users\apll1\AppData\Local\Temp\ipykernel_32024\290482630.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include="object").columns
C:\Users\apll1\AppData\Local\Temp\ipykernel_32024\290482630.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m